# My notes: Tree-Based ML Models (Wine dataset)

Personal reference notebook. Whenever I revisit this, the goal is to remember not just *what* the code does, but *why* I'm doing it at that point in the workflow.

**The shape of the whole thing, in one line:**
`build -> train -> inspect -> predict -> evaluate -> tune (pre-pruning / post-pruning) -> ensemble (forest / boosting)`

Every ML project I do with tabular data will basically follow this shape. If I'm lost later, come back to this line first.

## 0. Setup

**Why this cell exists:** nothing works without these imports. `xgboost` and `lightgbm` are NOT part of sklearn — they're separate libraries that just copy sklearn's `.fit()`/`.predict()` style so they slot into the same pipeline.

In [ ]:
# one-time install for the two libraries that aren't built into sklearn
!pip install -q xgboost lightgbm

import numpy as np
import matplotlib.pyplot as plt

# sklearn is organized into modules (folders of related tools) -
# tree = single-tree tools, ensemble = "team of trees" tools, metrics = grading tools
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import tree, ensemble, metrics
import xgboost as xgb
import lightgbm as lgb
import time

# silence noisy warnings so the output stays readable
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Load data + train/test split

**Why this matters:** never test on rows the model already saw during training — that's like grading a student on homework they copied. `train_test_split` keeps 20% completely hidden from training, as a fair "pop quiz."

**Remember:** `random_state` = a seed number, just makes the split (and later, the tree's tie-breaking) reproducible. Same seed = same result every time I re-run this.

In [ ]:
wine = load_wine()

X = wine.data            # table of chemical measurements (the "questions")
y = wine.target          # correct answers: which of 3 grape varieties (0, 1, 2)
feature_names = wine.feature_names
class_names = wine.target_names

seed = 4000               # my fixed seed for this whole notebook - keep consistent so results are comparable across sections
min_samples_leaf = 2      # don't create a leaf with fewer than this many samples (stops overly-specific rules)
min_samples_split = 3     # don't even try to split a node with fewer than this many samples

# 80% train ("homework"), 20% test ("quiz", never trained on)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed
)

print(f"Training rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")


## 2. Build + train a baseline Decision Tree

**Key distinction to remember:** *building* a model (creating the object) is not the same as *training* it. Building just configures settings — no data involved yet. `.fit()` is the step where it actually looks at data and grows the tree.

**Also remember:** `.fit()` mutates `clf` in place - it doesn't return a new object, it fills in the same one.

In [ ]:
clf = tree.DecisionTreeClassifier(
    criterion='gini',       # how to judge "which question is best" at each split (impurity measure)
    max_depth=None,         # no depth limit -> without this, min_samples_leaf/split are my only overfitting guardrails
    min_samples_leaf=min_samples_leaf,
    min_samples_split=min_samples_split,
    random_state=seed
)

clf.fit(X_train, y_train)   # <- this is the actual "learning" step. clf is now filled in, mutated in place.

print(clf)


## 3. Read the tree's actual learned rules

**Why I do this:** sanity check - does the tree's logic look reasonable, or is it doing something weird? This is a read-only inspection step, doesn't change `clf` at all.

In [ ]:
# export_text walks the tree's internal structure and prints it as nested if/else text
print(tree.export_text(
    clf,
    feature_names=list(feature_names),
    class_names=[str(c) for c in class_names]   # cast to str defensively - export_text wants plain strings
))


## 4. Predict + evaluate the baseline

**Reminder:** `.predict()` is read-only, unlike `.fit()`. It doesn't touch `clf`, it just returns a fresh array of guesses.

**How to read the report:** precision = "when I said class X, was I right?", recall = "of all the real class X, how many did I catch?", f1 = blend of both. Watch for classes with a big precision/recall gap - that's usually where two classes are getting confused with each other.

In [ ]:
y_pred = clf.predict(X_test)   # walk each test row down the tree, get the leaf's answer

print("Baseline Decision Tree -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))

# <- my baseline number to beat. Everything below should try to improve on this.


## 5. Pre-pruning via GridSearchCV

**The idea:** instead of guessing min_samples_leaf/split by hand (like I did above), let the computer try every combination from a grid and pick the winner using cross-validation (so it's not just overfitting to one lucky split).

**Gotcha to remember:** call `.fit()` on `grid_search`, NOT on the raw tree - the raw tree never gets touched directly, GridSearchCV handles building/training/scoring all combos internally.

In [ ]:
# every combination of these gets tried (2 x 3 x 3 x 3 = 54 trees total)
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': list(range(2, 5)),          # range(2,5) = 2,3,4 (5 excluded)
    'min_samples_leaf': list(range(2, 5)),
    'min_samples_split': list(range(2, 5)),
}

grid_search = GridSearchCV(estimator=tree.DecisionTreeClassifier(random_state=seed), param_grid=param_grid)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_   # trailing underscore = "only exists after .fit() ran"
print("Best pre-pruning parameters:", best_params)

# ** unpacks the dict into keyword args, e.g. criterion='gini', max_depth=4, ...
prepruned_clf = tree.DecisionTreeClassifier(**best_params, random_state=seed)
prepruned_clf.fit(X_train, y_train)

y_pred = prepruned_clf.predict(X_test)
print("\nPre-Pruned Decision Tree -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))

# compare this accuracy to my baseline above - should be equal or better


## 6. Post-pruning, part 1: get the pruning path

**Different philosophy from pre-pruning:** let the tree grow FULLY (overfit on purpose), then cut branches back afterward.

**ccp_alpha = my "strictness dial"** for cutting: 0 = no pruning (full tree), higher = prune more aggressively. `cost_complexity_pruning_path` finds every meaningful dial value automatically instead of me guessing one.

**What the plot should look like:** impurity rises as alpha rises - makes sense, pruning merges previously-pure leaves back together.

In [ ]:
full_clf = tree.DecisionTreeClassifier(
    min_samples_leaf=min_samples_leaf,
    min_samples_split=min_samples_split,
    random_state=seed
)
# note: this method needs X,y because it has to grow the full tree first to find the breakpoints
path = full_clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

plt.figure(figsize=(8, 6))
plt.plot(ccp_alphas, impurities, marker='o', drawstyle='steps-post')
plt.xlabel("ccp_alpha")
plt.ylabel("Total Leaf Impurity")
plt.title("Total Leaf Impurity vs ccp_alpha on Training Data")
plt.grid(True)
plt.show()


## 7. Post-pruning, part 2: train one tree per alpha

**This is the important plot to remember:** training accuracy should steadily DROP as alpha increases (simpler tree = worse memorization). Test accuracy should go UP first (pruning removes overfit noise), peak somewhere in the middle, then COLLAPSE at high alpha (too much pruning = underfitting).

That rise-then-fall on the test curve = the bias-variance tradeoff, visually. The peak is the sweet spot I want.

In [ ]:
clfs, train_scores, test_scores = [], [], []

for ccp_alpha in ccp_alphas:
    c = tree.DecisionTreeClassifier(
        random_state=seed,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        ccp_alpha=ccp_alpha
    )
    c.fit(X_train, y_train)
    train_scores.append(c.score(X_train, y_train))  # .score() = shortcut for accuracy, no need to call predict() separately
    test_scores.append(c.score(X_test, y_test))
    clfs.append(c)

plt.figure(figsize=(8, 6))
plt.plot(ccp_alphas, train_scores, marker='o', label='Train Accuracy', drawstyle='steps-post')
plt.plot(ccp_alphas, test_scores, marker='o', label='Test Accuracy', drawstyle='steps-post')
plt.xlabel('ccp_alpha')
plt.ylabel('Accuracy')
plt.title('Effect of ccp_alpha on Train and Test Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## 8. Post-pruning, part 3: pick the actual winner

**The 3-step tiebreak rule I'm using (in priority order):**
1. highest test accuracy
2. if tied -> smallest |train - test| gap (favors the less-overfit tree)
3. if still tied -> biggest ccp_alpha (favors the simpler tree - Occam's razor)

**How the code does all 3 steps in one line:** build a tuple `(test_score, -gap, alpha)` per candidate. Python compares tuples left-to-right, so `max()` on these tuples naturally implements the cascading rule. Negating the gap flips "smaller is better" into "bigger is better" so everything stays consistent for `max()`.

In [ ]:
def sort_key(i):
    gap = abs(train_scores[i] - test_scores[i])
    return (test_scores[i], -gap, ccp_alphas[i])

best_index = max(range(len(clfs)), key=sort_key)
best_alpha = ccp_alphas[best_index]
best_pruned_clf = clfs[best_index]

print(f"Best alpha: {best_alpha}")

y_pred = best_pruned_clf.predict(X_test)
print("\nPost-Pruned Decision Tree -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))

# compare this to both the baseline AND the pre-pruned version above


## 9. Random Forest

**Mental model:** stop trying to perfect ONE tree - instead train MANY trees and let them vote (majority wins). Diversity between trees comes from two sources of randomness:
- each tree sees a random bootstrap sample of rows
- each split only considers a random subset of features

**Why this usually beats a single pruned tree:** individual trees' mistakes tend to cancel out when averaged/voted across many diverse trees.

**Reminder:** RandomForestClassifier has the same `.fit()`/`.predict()` shape as DecisionTreeClassifier - that's why nothing else in my code needs to change to use it.

In [ ]:
rf = ensemble.RandomForestClassifier(n_estimators=20, random_state=seed)  # n_estimators = how many trees in the forest
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))

# this should be my best accuracy so far - forests usually beat single trees, pruned or not


## 10. Feature importance

**What this tells me:** which columns did the forest actually rely on? A trained forest tracks this automatically (from how much each feature reduced impurity, averaged across all trees) - it's free interpretability, no extra work needed.

**Why I sort with np.argsort:** it gives me the *order of indices* that would sort importances ascending, not the sorted values themselves - I need that so I can reorder feature_names to match, since barh draws bottom-to-top (so ascending order puts the most important feature visually on top).

In [ ]:
importances = rf.feature_importances_   # one score per feature, all sum to 1
indices = np.argsort(importances)         # ascending order of indices

plt.figure(figsize=(10, 6))
plt.barh(range(len(importances)), importances[indices])
plt.yticks(range(len(importances)), [feature_names[i] for i in indices])
plt.xlabel("Feature Importance")
plt.title("Random Forest - Feature Importances")
plt.tight_layout()
plt.show()

# whichever feature is at the top of this chart is what's driving most of the forest's decisions


## 11. Boosting: AdaBoost / GradientBoosting / XGBoost / LightGBM

**How this is different from Random Forest:** forest trees are independent (parallel, vote at the end). Boosting trees are sequential - each new tree is trained specifically to fix the previous trees' mistakes, then all outputs get combined (weighted).

**Tradeoff to remember:** because each tree depends on the last, boosting usually can't be parallelized the way bagging can - so it's often more accurate but slower to train. That's exactly what I'm measuring below (accuracy AND training time).

**Note to self:** `verbose=-1` on LightGBM specifically silences its default wall of training logs - just a library quirk, not a general rule.

In [ ]:
def build_boosting_model(name, n_estimators, random_state):
    """Factory function: given a name string, return the matching untrained boosting model."""
    if name == "adaboost":
        return ensemble.AdaBoostClassifier(n_estimators=n_estimators, random_state=random_state)
    elif name == "gradientboosting":
        return ensemble.GradientBoostingClassifier(n_estimators=n_estimators, random_state=random_state)
    elif name == "xgboost":
        return xgb.XGBClassifier(n_estimators=n_estimators, random_state=random_state)
    elif name == "lightgbm":
        return lgb.LGBMClassifier(n_estimators=n_estimators, random_state=random_state, verbose=-1)
    else:
        raise ValueError(f"Unknown model name '{name}'")

model_names = ["adaboost", "gradientboosting", "xgboost", "lightgbm"]
n_estimators = 20
n_runs = 50  # lowered from 500 so this cell runs faster in Colab - bump up for a more stable timing average

accuracies = {}
avg_train_times = {}

for name in model_names:
    # time training over many runs and average, since a single run can be noisy (system load etc.)
    times = []
    for _ in range(n_runs):
        model = build_boosting_model(name, n_estimators, seed)
        start = time.time()
        model.fit(X_train, y_train)
        times.append(time.time() - start)
    avg_train_times[name] = sum(times) / n_runs

    # one more clean fit just to measure accuracy
    model = build_boosting_model(name, n_estimators, seed)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracies[name] = metrics.accuracy_score(y_test, y_pred)

print("Accuracies:", accuracies)
print("Avg training times (s):", avg_train_times)


In [ ]:
colors = ['blue', 'orange', 'green', 'red']

plt.figure(figsize=(9, 5))
plt.bar(accuracies.keys(), accuracies.values(), color=colors)
plt.title("Accuracy Comparison of Boosting Classifiers on the Wine Dataset")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
for i, v in enumerate(accuracies.values()):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center', fontweight='bold')
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(avg_train_times.keys(), avg_train_times.values(), color=colors)
plt.ylabel("Time (seconds)")
plt.title(f"Average Training Time ({n_runs} runs) for Boosting Classifiers")
for i, v in enumerate(avg_train_times.values()):
    plt.text(i, v + 0.001, f"{v:.3f}", ha='center', fontweight='bold')
plt.show()

# read this side by side with the accuracy chart above - the "best" model depends on
# whether I care more about accuracy or training speed for the situation


## My takeaways (fill in / update each time I revisit)

- Baseline tree accuracy: ~0.75 -> a single unlimited-depth tree overfits, this is the number to beat
- Pre-pruned tree: grid search found better stopping rules than my hand-picked guesses -> accuracy went up
- Post-pruned tree: same idea, different method (grow full then cut back) -> similar improvement, found via ccp_alpha
- Random Forest: biggest jump, ~0.94 -> voting across many diverse trees beats any single tuned tree
- Boosting: compare accuracy vs training time trade-off per algorithm, pick based on what the actual project needs (accuracy-critical vs needs-to-be-fast)

**General lesson I keep relearning:** don't reach for the fanciest model first. Start simple (single tree), understand *why* it's underperforming (overfitting), THEN reach for pruning or ensembling to fix that specific problem. Model complexity should follow the diagnosis, not precede it.